In [12]:
pip install rdflib networkx gensim scikit-learn numpy


Note: you may need to restart the kernel to use updated packages.


In [13]:
import csv
import random
from collections import Counter
from pathlib import Path
from typing import Iterable

import networkx as nx
import numpy as np
from gensim.models import Word2Vec
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import OWL, RDF, RDFS
from sklearn.metrics.pairwise import cosine_similarity



1. Load the files

In [14]:
HI = Namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")

TTL_FILES = [
    "hi_ontology.ttl",
    "hi_extension.ttl",
    "hi_instances.ttl",
]

OUTPUT_DIR = Path("kg_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def load_graph(files: Iterable[str]) -> Graph:
    g = Graph()
    for file in files:
        path = Path(file)
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {file}")
        g.parse(path, format="turtle")
    return g

2. SPARQL Queries

In [15]:
def run_query(graph: Graph, query: str):
    return list(graph.query(query))


def save_query_results(rows, headers, filepath: Path) -> None:
    with filepath.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for row in rows:
            writer.writerow([str(x) for x in row])


def short_uri(value) -> str:
    value = str(value)
    if "#" in value:
        return value.split("#")[-1]
    return value.split("/")[-1]


SPARQL_QUERIES = {
    "q1_papers_per_scenario": {
        "description": "Count how many papers are linked to each main scenario.",
        "headers": ["scenario", "paperCount"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?scenario (COUNT(DISTINCT ?paper) AS ?paperCount)
        WHERE {
          ?paper a hi:ResearchPaper ;
                 hi:relevantToScenario ?scenario .
        }
        GROUP BY ?scenario
        ORDER BY DESC(?paperCount)
        """
    },
    "q2_cross_domain_papers": {
        "description": "Find papers linked to more than one scenario.",
        "headers": ["paper", "scenarioCount"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?paper (COUNT(DISTINCT ?scenario) AS ?scenarioCount)
        WHERE {
          ?paper a hi:ResearchPaper ;
                 hi:relevantToScenario ?scenario .
        }
        GROUP BY ?paper
        HAVING (COUNT(DISTINCT ?scenario) > 1)
        ORDER BY DESC(?scenarioCount)
        """
    },
    "q3_trust_states": {
        "description": "List all entities with trust states.",
        "headers": ["entity", "trustState"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?entity ?trustState
        WHERE {
          ?entity hi:hasTrustState ?trustState .
        }
        ORDER BY ?entity
        """
    },
    "q4_autonomy_levels": {
        "description": "Count autonomy levels across the graph.",
        "headers": ["autonomyLevel", "count"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?autonomyLevel (COUNT(?x) AS ?count)
        WHERE {
          ?x hi:hasAutonomyLevel ?autonomyLevel .
        }
        GROUP BY ?autonomyLevel
        ORDER BY DESC(?count)
        """
    },
    "q5_ethics_per_main_scenario": {
        "description": "List ethical considerations for each main scenario.",
        "headers": ["scenario", "ethics"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?scenario (GROUP_CONCAT(REPLACE(STR(?ethic), "^.*[#/]", ""); separator=", ") AS ?ethics)
        WHERE {
          ?scenario hi:hasEthicalConsideration ?ethic .
          FILTER NOT EXISTS { ?scenario a hi:ScenarioVariant . }
        }
        GROUP BY ?scenario
        ORDER BY ?scenario
        """
    },
    "q6_actor_capabilities_per_scenario": {
        "description": "List actors and their capabilities per scenario.",
        "headers": ["scenario", "actor", "capabilities"],
        "query": """
        PREFIX hi: <http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/>

        SELECT ?scenario ?actor (GROUP_CONCAT(REPLACE(STR(?cap), "^.*[#/]", ""); separator=", ") AS ?capabilities)
        WHERE {
          ?scenario hi:involvesActor ?actor .
          OPTIONAL { ?actor hi:capability ?cap . }
        }
        GROUP BY ?scenario ?actor
        ORDER BY ?scenario ?actor
        """
    }
}



3. Compute Metrics

In [16]:
def count_query(graph: Graph, query: str) -> int:
    return int(list(graph.query(query))[0][0])


def compute_basic_metrics(graph: Graph) -> dict:
    metrics = {}

    metrics["total_triples"] = len(graph)

    metrics["classes"] = count_query(graph, """
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        SELECT (COUNT(DISTINCT ?c) AS ?count)
        WHERE { ?c a owl:Class . }
    """)

    metrics["object_properties"] = count_query(graph, """
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        SELECT (COUNT(DISTINCT ?p) AS ?count)
        WHERE { ?p a owl:ObjectProperty . }
    """)

    metrics["datatype_properties"] = count_query(graph, """
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        SELECT (COUNT(DISTINCT ?p) AS ?count)
        WHERE { ?p a owl:DatatypeProperty . }
    """)

    metrics["named_individuals"] = count_query(graph, """
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        SELECT (COUNT(DISTINCT ?i) AS ?count)
        WHERE { ?i a owl:NamedIndividual . }
    """)

    metrics["research_papers"] = count_query(graph, f"""
        PREFIX hi: <{HI}>
        SELECT (COUNT(DISTINCT ?x) AS ?count)
        WHERE {{ ?x a hi:ResearchPaper . }}
    """)

    metrics["scenarios"] = count_query(graph, f"""
        PREFIX hi: <{HI}>
        SELECT (COUNT(DISTINCT ?x) AS ?count)
        WHERE {{ ?x a hi:Scenario . }}
    """)

    metrics["scenario_variants"] = count_query(graph, f"""
        PREFIX hi: <{HI}>
        SELECT (COUNT(DISTINCT ?x) AS ?count)
        WHERE {{ ?x a hi:ScenarioVariant . }}
    """)

    metrics["actors"] = count_query(graph, f"""
        PREFIX hi: <{HI}>
        SELECT (COUNT(DISTINCT ?x) AS ?count)
        WHERE {{ ?x a hi:Actor . }}
    """)

    external_links = 0
    for _, _, o in graph:
        if isinstance(o, URIRef):
            o_str = str(o)
            if (
                o_str.startswith("https://dbpedia.org/")
                or o_str.startswith("http://dbpedia.org/")
                or o_str.startswith("https://www.wikidata.org/")
                or o_str.startswith("http://www.wikidata.org/")
                or o_str.startswith("https://schema.org/")
                or o_str.startswith("http://purl.org/ontology/")
                or o_str.startswith("http://xmlns.com/foaf/")
            ):
                external_links += 1
    metrics["external_links"] = external_links

    return metrics


def build_network(graph: Graph) -> nx.DiGraph:
    dg = nx.DiGraph()
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            dg.add_edge(str(s), str(o), predicate=str(p))
    return dg


def compute_network_metrics(graph: Graph) -> tuple[dict, list[tuple[str, int]], list[tuple[str, int]]]:
    dg = build_network(graph)

    metrics = {
        "iri_nodes": dg.number_of_nodes(),
        "iri_edges": dg.number_of_edges(),
        "density": nx.density(dg),
    }

    top_nodes = sorted(dg.degree, key=lambda x: x[1], reverse=True)[:15]

    pred_counter = Counter()
    for _, _, p_data in dg.edges(data=True):
        pred_counter[short_uri(p_data["predicate"])] += 1
    top_predicates = pred_counter.most_common(15)

    return metrics, top_nodes, top_predicates


4. Run embeddings / simple link prediction

In [17]:
def get_nodes_by_type(graph: Graph, class_uri: URIRef) -> list[str]:
    return [str(row[0]) for row in graph.query(
        """
        SELECT ?x WHERE { ?x a ?cls . }
        """,
        initBindings={"cls": class_uri}
    )]


def build_adjacency(graph: Graph) -> dict[str, list[tuple[str, str]]]:
    adj: dict[str, list[tuple[str, str]]] = {}
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            s_str = str(s)
            p_str = str(p)
            o_str = str(o)
            adj.setdefault(s_str, []).append((p_str, o_str))
            adj.setdefault(o_str, []).append((p_str + "_inv", s_str))
    return adj


def make_random_walks(adj: dict[str, list[tuple[str, str]]], starts: list[str], num_walks: int = 80, depth: int = 6) -> list[list[str]]:
    walks = []

    for start in starts:
        for _ in range(num_walks):
            current = start
            walk = [short_uri(current)]
            for _ in range(depth):
                neighbors = adj.get(current, [])
                if not neighbors:
                    break
                pred, nxt = random.choice(neighbors)
                walk.extend([short_uri(pred), short_uri(nxt)])
                current = nxt
            walks.append(walk)

    return walks


def train_embeddings(walks: list[list[str]]) -> Word2Vec:
    return Word2Vec(
        sentences=walks,
        vector_size=64,
        window=5,
        min_count=1,
        sg=1,
        workers=1,
        epochs=50,
        seed=RANDOM_SEED
    )


def predict_missing_paper_scenario_links(graph: Graph, model: Word2Vec) -> list[tuple[float, str, str]]:
    papers = get_nodes_by_type(graph, HI.ResearchPaper)

    scenarios = [str(row[0]) for row in graph.query(f"""
        PREFIX hi: <{HI}>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

        SELECT DISTINCT ?s
        WHERE {{
          ?s a ?t .
          ?t rdfs:subClassOf* hi:Scenario .
          FILTER(?t != hi:ScenarioVariant)
        }}
    """)]

    existing_links = set(
        (str(row[0]), str(row[1]))
        for row in graph.query(f"""
            PREFIX hi: <{HI}>
            SELECT ?paper ?scenario
            WHERE {{
              ?paper a hi:ResearchPaper ;
                     hi:relevantToScenario ?scenario .
            }}
        """)
    )

    candidates = []
    for p in papers:
        for s in scenarios:
            if (p, s) in existing_links:
                continue
            p_key = short_uri(p)
            s_key = short_uri(s)
            if p_key in model.wv and s_key in model.wv:
                score = float(cosine_similarity([model.wv[p_key]], [model.wv[s_key]])[0][0])
                candidates.append((score, p_key, s_key))

    candidates.sort(reverse=True)
    return candidates[:15]


def most_similar_papers(graph: Graph, model: Word2Vec, target_paper: str) -> list[tuple[str, float]]:
    results = []
    for node, score in model.wv.most_similar(target_paper, topn=20):
        if node.startswith("Paper"):
            results.append((node, float(score)))
    return results[:10]

5. Print Results

In [18]:
def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def print_table(rows, headers) -> None:
    if not rows:
        print("(no results)")
        return

    str_rows = [[str(x) for x in row] for row in rows]
    widths = [len(h) for h in headers]

    for row in str_rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(cell))

    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

    print(header_line)
    print(sep_line)
    for row in str_rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(headers))))


6. Save results to files

In [19]:
def save_metrics(metrics: dict, filepath: Path) -> None:
    with filepath.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in metrics.items():
            writer.writerow([k, v])


def save_list_rows(rows: list[tuple], headers: list[str], filepath: Path) -> None:
    with filepath.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for row in rows:
            writer.writerow(row)


7. Main

In [20]:
def main() -> None:
    print_section("STEP 1 - LOADING TTL FILES")
    graph = load_graph(TTL_FILES)
    print(f"Loaded graph with {len(graph)} triples from {len(TTL_FILES)} files.")

    print_section("STEP 2 - RUNNING SPARQL QUERIES")
    for name, info in SPARQL_QUERIES.items():
        print(f"\n{name}: {info['description']}")
        rows = run_query(graph, info["query"])
        print_table(rows, info["headers"])
        save_query_results(rows, info["headers"], OUTPUT_DIR / f"{name}.csv")

    print_section("STEP 3 - COMPUTING KG METRICS")
    basic_metrics = compute_basic_metrics(graph)
    for k, v in basic_metrics.items():
        print(f"{k}: {v}")
    save_metrics(basic_metrics, OUTPUT_DIR / "basic_metrics.csv")

    network_metrics, top_nodes, top_predicates = compute_network_metrics(graph)

    print("\nNetwork metrics:")
    for k, v in network_metrics.items():
        print(f"{k}: {v}")
    save_metrics(network_metrics, OUTPUT_DIR / "network_metrics.csv")

    print("\nTop nodes by degree:")
    top_nodes_clean = [(short_uri(n), d) for n, d in top_nodes]
    print_table(top_nodes_clean, ["node", "degree"])
    save_list_rows(top_nodes_clean, ["node", "degree"], OUTPUT_DIR / "top_nodes.csv")

    print("\nTop predicates:")
    print_table(top_predicates, ["predicate", "count"])
    save_list_rows(top_predicates, ["predicate", "count"], OUTPUT_DIR / "top_predicates.csv")

    print_section("STEP 4 - TRAINING EMBEDDINGS / LINK PREDICTION")
    papers = get_nodes_by_type(graph, HI.ResearchPaper)
    scenarios = [str(row[0]) for row in graph.query(f"""
        PREFIX hi: <{HI}>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

        SELECT DISTINCT ?s
        WHERE {{
          ?s a ?t .
          ?t rdfs:subClassOf* hi:Scenario .
          FILTER(?t != hi:ScenarioVariant)
        }}
    """)]

    starts = papers + scenarios
    adj = build_adjacency(graph)
    walks = make_random_walks(adj, starts, num_walks=80, depth=6)
    print(f"Generated {len(walks)} random walks.")

    model = train_embeddings(walks)
    print("Embedding model trained.")

    print_section("STEP 5 - PRINTING SUBSYMBOLIC RESULTS")
    predicted_links = predict_missing_paper_scenario_links(graph, model)
    print("Top predicted missing paper-scenario links:")
    print_table([(round(score, 3), paper, scenario) for score, paper, scenario in predicted_links],
                ["score", "paper", "scenario"])
    save_list_rows(
        [(round(score, 6), paper, scenario) for score, paper, scenario in predicted_links],
        ["score", "paper", "scenario"],
        OUTPUT_DIR / "predicted_missing_links.csv"
    )

    target = "Paper10_BRIDGET"
    if target in model.wv:
        similar = most_similar_papers(graph, model, target)
        print(f"\nMost similar papers to {target}:")
        print_table([(paper, round(score, 3)) for paper, score in similar], ["paper", "similarity"])
        save_list_rows(
            [(paper, round(score, 6)) for paper, score in similar],
            ["paper", "similarity"],
            OUTPUT_DIR / "similar_to_paper10_bridget.csv"
        )
    else:
        print(f"\nTarget paper {target} not found in embedding vocabulary.")

    print_section("STEP 6 - SAVING RESULTS")
    print(f"All CSV files saved in: {OUTPUT_DIR.resolve()}")

    print("\nDone.")


if __name__ == "__main__":
    main()


STEP 1 - LOADING TTL FILES
Loaded graph with 810 triples from 3 files.

STEP 2 - RUNNING SPARQL QUERIES

q1_papers_per_scenario: Count how many papers are linked to each main scenario.
scenario                                                                                         | paperCount
-------------------------------------------------------------------------------------------------+-----------
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/HealthcareDiagnosisTeam | 9         
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/ChildEducationTeam      | 5         
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/ResearchHITeam          | 2         

q2_cross_domain_papers: Find papers linked to more than one scenario.
paper                                                                                                    | scenarioCount
------------------------------------------------------------------------